<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/GLM_Caso2_Informe_Ejecutivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

equipo	N	m_sin	disp_sin	p0_fraude	p_evento	p_baja	m_coste
equipo_07	260107	1.1	1.1	1	1.1	0.7	1.7	1.05	0.56


In [ ]:
library(tidyverse)

# Fuente de datos del equipo 07
repositorio <- "https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/"

cartera <- read_csv(
    paste0(repositorio, "cartera.csv"),
    show_col_types = FALSE
    )

# Comprobar que los datos son válidos
stopifnot(
    nrow(cartera) > 0,
    all(c("n_siniestros", "n_asistencia", "exposicion")    %in% names(cartera)),
    all(!is.na(cartera$exposicion)),
    all(cartera$exposicion > 0)
    )

glimpse(cartera)
summary(cartera$exposicion)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Rows: 3,062
Columns: 40
$ id_poliza           <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,…
$ id_agencia          <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ id_region           <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ antiguedad          <dbl> 1.082, 6.687, 5.732, 1.397, 1.968, 3.159, 3.821, 7…
$ exposicion          <dbl> 0.1082, 0.6687, 0.5732, 0.1397, 0.1968, 0.3159, 0.…
$ edad_conductor      <dbl> 33, 44, 62, 25, 66, 25, 18, 44, 49, 36, 77, 36, 39…
$ edad_tramo          <chr> "25-35", "35-65", "35-65", "18-25", "65+", "18-25"…
$ antiguedad_carnet   <dbl> 7, 13, 36, 6, 36, 7, 0, 11, 30, 7, 49, 12, 16, 17,…
$ potencia_cv         <dbl> 85, 142, 118, 76, 55, 120, 143, 64, 113, 136, 141,…
$ cilindrada          <dbl> 1417, 2423, 1831, 1278, 824, 1816, 1730, 1006, 162…
$ antiguedad_vehiculo <dbl> 6, 12, 6, 9, 10, 6, 11, 10, 19, 4, 7, 12, 19, 4, 7…
$ valor_vehiculo      <dbl> 35790, 22089, 31528, 25948, 20714, 15821, 23040, 2…
$ densidad      

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0100  0.2194  0.3768  0.4207  0.5934  0.9996 

# GLM · Caso 2 · Informe ejecutivo · Equipo 07-Cox

## Integración del código de la profesora

Se han incorporado **celdas R reales**, adaptando los métodos de los scripts de [Asun Mayoral: caso2/scripts](https://github.com/asunmayoral/GLM/tree/master/caso2/scripts). No se importan ni ejecutan los scripts remotos: **todo el código analítico de este informe está escrito en este mismo notebook**.

Los ejemplos de la profesora usan otra cartera simulada, una variable llamada n_danos y un panel temporal que no son los datos del equipo 07. Aquí usamos **n_siniestros** como variable de recuento del equipo, no como una equivalencia demostrada con n_danos. También reemplazamos la unidad longitudinal 2.6 por un modelo de **tiempo hasta el primer siniestro**: ambos análisis responden preguntas diferentes.

**Ruta del informe:** presentación de la cartera → tasas Poisson (2.1) → contingencia y log-lineales (2.2) → sobredispersión (2.3) → modelos de ceros (2.4) → efectos aleatorios (2.5) → riesgo del primer siniestro (adaptación de 2.6) → informe y decisiones (2.7).

Las secciones 2.4–2.6 requieren paquetes opcionales; se indican de forma explícita. No se utiliza ninguna “verdad del generador de datos” (DGP) porque no está disponible para la cartera observada. Los resultados se obtendrán **al ejecutar las celdas**, sin cifras inventadas.

### 🔰 Dependencias y comprobación

Ejecuta primero la carga de datos original que aparece arriba. La siguiente celda **no instala nada ni crea archivos**, solo comprueba los paquetes. Si faltan, la línea de instalación está preparada y comentada para ejecutarla una vez.

In [ ]:
paquetes_glm <- c(
  "tidyverse", "MASS", "broom", "glmmTMB",
  "DHARMa", "performance", "survival"
)
faltantes_glm <- paquetes_glm[
  !vapply(paquetes_glm, requireNamespace, logical(1), quietly = TRUE)
]
if (length(faltantes_glm)) {
  message("Paquetes opcionales pendientes: ", paste(faltantes_glm, collapse = ", "))
  # install.packages(faltantes_glm, repos = "https://cloud.r-project.org")
} else {
  message("Todos los paquetes están disponibles.")
}

set.seed(20252026L)
theme_set(ggplot2::theme_minimal(base_size = 14))

### 0 · Datos comunes a todas las unidades

Conservamos la tabla original **cartera** y construimos **datos_07** para los modelos. Las agencias se identifican conjuntamente por región y agencia, evitando confundir identificadores locales que se repitan entre regiones. Comprobamos tipos, exposición y ausencia de datos obligatorios.

**Unidades:** antiguedad mide años; exposicion equivale aproximadamente a años/10. Para el encargo se usa offset(log(antiguedad)), según la sección 7.1 de las notas.

In [ ]:
columnas_glm <- c(
  "id_poliza", "id_agencia", "id_region", "exposicion",
  "edad_conductor", "potencia_cv", "zona_circulacion",
  "uso", "tipo_vehiculo", "antiguedad",
  "n_siniestros", "n_asistencia", "n_fraude",
  "n_gestiones", "t_primer_sin", "evento"
)
stopifnot(
  all(columnas_glm %in% names(cartera)),
  all(is.finite(cartera$exposicion)),
  all(cartera$exposicion > 0),
  all(is.finite(cartera$antiguedad)),
  all(cartera$antiguedad > 0),
  all(cartera$tuvo_siniestro == as.integer(cartera$n_siniestros > 0)),
  all(!is.na(cartera$n_siniestros)),
  all(!is.na(cartera$n_asistencia)),
  all(!is.na(cartera$n_fraude)),
  all(!is.na(cartera$n_gestiones)),
  all(cartera$n_siniestros >= 0),
  all(cartera$n_fraude >= 0),
  !anyDuplicated(cartera$id_poliza)
)

datos_07 <- cartera |>
  dplyr::mutate(
    region = factor(id_region),
    agencia = interaction(id_region, id_agencia, drop = TRUE),
    zona_circulacion = factor(zona_circulacion),
    uso = factor(uso),
    tipo_vehiculo = factor(tipo_vehiculo),
    potencia_c = potencia_cv - mean(potencia_cv),
    asistencia = factor(
      ifelse(n_asistencia > 0, "con", "sin"), levels = c("sin", "con")
    ),
    siniestro = factor(
      ifelse(n_siniestros > 0, "con", "sin"), levels = c("sin", "con")
    )
  )

cat("Pólizas:", nrow(datos_07),
    "| Regiones:", dplyr::n_distinct(datos_07$region),
    "| Agencias:", dplyr::n_distinct(datos_07$agencia), "\n")
summary(datos_07$antiguedad)

## Presentación · ¿Cuántas veces y a qué ritmo?

**Adaptado de caso2_conteos.R, bloques tbl-cartera-resp y fig-cartera-eda.** La media de recuentos por póliza no equivale a la tasa por unidad de exposición. Mostramos ambas antes de formular modelos.

La tasa presentada para el informe está expresada en años usando antiguedad. Las otras respuestas se muestran solo como contexto didáctico.

In [ ]:
respuestas_07 <- c(
  "n_asistencia", "n_siniestros", "n_fraude", "n_gestiones"
)

resumen_respuestas <- purrr::map_dfr(respuestas_07, function(variable) {
  y <- datos_07[[variable]]
  tibble::tibble(
    respuesta = variable,
    polizas = length(y),
    eventos = sum(y),
    media_poliza = mean(y),
    tasa_anual = sum(y) / sum(datos_07$antiguedad),
    proporcion_ceros = mean(y == 0),
    varianza_media = if (mean(y) > 0) var(y) / mean(y) else NA_real_
  )
})

resumen_respuestas |>
  dplyr::mutate(dplyr::across(where(is.numeric), ~round(.x, 3)))

In [ ]:
datos_07 |>
  dplyr::select(dplyr::all_of(respuestas_07)) |>
  tidyr::pivot_longer(dplyr::everything(), names_to = "respuesta",
                      values_to = "conteo") |>
  dplyr::mutate(tramo = pmin(conteo, 8L)) |>
  ggplot2::ggplot(ggplot2::aes(x = factor(tramo))) +
  ggplot2::geom_bar(fill = "goldenrod", color = "grey25") +
  ggplot2::facet_wrap(~respuesta, scales = "free_y") +
  ggplot2::labs(
    x = "Recuento (8 = ocho o más)",
    y = "Pólizas", title = "Distribución de las respuestas · Equipo 07"
  ) +
  ggplot2::theme(axis.text.x = ggplot2::element_text(size = 11))

## 2.1 · Poisson: frecuencia anual de siniestros (P2, modelo base)

Basado en la unidad 2.1 de la profesora: Poisson, IRR, predicción e índice de dispersión. Para el encargo de 2.7 se usa **n_siniestros** y **antigüedad en años**.

E[n_siniestros | x] = antiguedad × exp(β₀ + βx). El offset log(antiguedad) lleva coeficiente fijo 1. Los IRR = exp(β) son razones de tasas anuales. La adecuación de Poisson se comprueba en 2.3.

In [ ]:
# Datos de la cartera del equipo 07. Tiempo = años de antigüedad.
formula_siniestros <- n_siniestros ~ edad_conductor + potencia_cv +
  zona_circulacion + uso + tipo_vehiculo + offset(log(antiguedad))
m_siniestros_pois <- stats::glm(
  formula_siniestros, family = stats::poisson(link = "log"), data = datos_07
)

resumen_coeficientes <- function(modelo) {
  coeficientes <- summary(modelo)$coefficients
  z <- stats::qnorm(0.975)
  tibble::tibble(
    termino = rownames(coeficientes),
    beta = coeficientes[, 1],
    IRR = exp(coeficientes[, 1]),
    IC_inf = exp(coeficientes[, 1] - z * coeficientes[, 2]),
    IC_sup = exp(coeficientes[, 1] + z * coeficientes[, 2]),
    p_valor = coeficientes[, 4]
  )
}
resumen_coeficientes(m_siniestros_pois)

In [ ]:
# Los cinco perfiles del enunciado de la profesora (unidad 2.7, tabla u27-perfil).
# Todos se predicen durante UN AÑO: antiguedad = 1.
perfiles_07 <- tibble::tibble(
  perfil = c("A · Joven urbano", "B · Familiar rural", "C · Moto ciudad",
             "D · Furgoneta comercial", "E · Conductor mayor"),
  edad_conductor = c(23, 45, 30, 40, 70),
  potencia_cv = c(110, 120, 70, 130, 90),
  zona_circulacion = factor(
    c("urbana", "rural", "urbana", "mixta", "mixta"),
    levels = levels(datos_07$zona_circulacion)
  ),
  uso = factor(
    c("particular", "particular", "particular", "comercial", "particular"),
    levels = levels(datos_07$uso)
  ),
  tipo_vehiculo = factor(
    c("turismo", "turismo", "moto", "furgoneta", "turismo"),
    levels = levels(datos_07$tipo_vehiculo)
  ),
  antiguedad = rep(1, 5)
)

prediccion_pois <- stats::predict(
  m_siniestros_pois, newdata = perfiles_07,
  type = "link", se.fit = TRUE
)
predicciones_07 <- tibble::tibble(
  perfil = perfiles_07$perfil,
  siniestros_esperados = as.numeric(exp(prediccion_pois$fit)),
  media_IC_inf = as.numeric(exp(prediccion_pois$fit - 1.96 * prediccion_pois$se.fit)),
  media_IC_sup = as.numeric(exp(prediccion_pois$fit + 1.96 * prediccion_pois$se.fit)),
  conteo_PI_inf = as.numeric(stats::qpois(0.025, lambda = exp(prediccion_pois$fit))),
  conteo_PI_sup = as.numeric(stats::qpois(0.975, lambda = exp(prediccion_pois$fit)))
)
predicciones_07

# IC: incertidumbre de media estimada. PI: variabilidad del recuento
# individual suponiendo Poisson; hay que revisar tras 2.3.

In [ ]:
dispersion_pearson <- function(modelo) {
  sum(stats::residuals(modelo, type = "pearson")^2) /
    stats::df.residual(modelo)
}

indice_dispersion_siniestros <- dispersion_pearson(m_siniestros_pois)
c(
  dispersion_pearson = indice_dispersion_siniestros,
  dispersion_deviance = stats::deviance(m_siniestros_pois) /
    stats::df.residual(m_siniestros_pois)
)

In [ ]:
# Rootograma discreto; la última clase acumula toda la cola.
observado <- datos_07$n_siniestros
mu <- stats::fitted(m_siniestros_pois)
K <- 0:min(8L, max(observado))
esperados <- vapply(K, function(k) {
  if (k == max(K)) sum(stats::ppois(k - 1L, mu, lower.tail = FALSE))
  else sum(stats::dpois(k, mu))
}, numeric(1))
observados <- vapply(K, function(k) {
  if (k == max(K)) sum(observado >= k) else sum(observado == k)
}, numeric(1))

tabla_rootograma <- tibble::tibble(
  conteo = factor(K),
  raiz_obs = sqrt(observados),
  raiz_esp = sqrt(esperados),
  diferencia = sqrt(observados) - sqrt(esperados)
)
ggplot2::ggplot(tabla_rootograma, ggplot2::aes(conteo, diferencia)) +
  ggplot2::geom_col(fill = "goldenrod", color = "grey30") +
  ggplot2::geom_hline(yintercept = 0, linetype = "dashed") +
  ggplot2::labs(x = "Nº siniestros", y = "√observado − √esperado",
                title = "Rootograma · Siniestros, ajuste Poisson")

## 2.2 · Tablas de contingencia y modelos log-lineales (P1)

**Revisado con la unidad 2.2 y el encargo de la profesora.** La respuesta para P1 es **tuvo_siniestro** (presencia de siniestro), no asistencia. Se utiliza una tabla de 3 vías: zona × vehículo × siniestro. Cada observación del GLM es una **celda**, por lo que **no** se añade offset de exposición en este modelo de frecuencias.

Estudiamos cinco hipótesis jerárquicas, incluida la independencia condicional. El saturado sirve como **referencia algebraica** (deviance 0), evitando ejecutar un ajuste innecesario con tantos parámetros como celdas.

Antes del contraste comprobamos cuántas celdas contienen menos de cinco pólizas. Los ceros muestrales no se eliminan como si fueran estructurales.

In [ ]:
# P1: siniestros (no asistencias). Una fila en la tabla = una combinación.
tabla_3v <- stats::xtabs(
  ~ zona_circulacion + tipo_vehiculo + siniestro,
  data = datos_07
)
frecuencias_3v <- as.data.frame(tabla_3v)

stopifnot(
  sum(frecuencias_3v$Freq) == nrow(datos_07),
  nrow(frecuencias_3v) == prod(dim(tabla_3v)),
  !anyNA(frecuencias_3v$Freq)
)

ftable(tabla_3v)
c(
  celdas = nrow(frecuencias_3v),
  vacias = sum(frecuencias_3v$Freq == 0),
  menores_de_5 = sum(frecuencias_3v$Freq < 5)
)

In [ ]:
# Cinco hipótesis de la profesora (unidad 2.2, u22-jerarquia).
# Cuatro GLM más el saturado calculado como referencia, sin ajustarlo.
ajustar_tabla <- function(estructura) {
  ajuste <- tryCatch(
    stats::glm(
      formula = estructura, family = stats::poisson(),
      data = frecuencias_3v,
      control = stats::glm.control(maxit = 100)
    ),
    error = function(error) {
      stop("Error ajustando ", deparse(estructura), ": ",
           conditionMessage(error), call. = FALSE)
    }
  )
  if (!isTRUE(ajuste$converged)) {
    stop("No converge el modelo ", deparse(estructura), call. = FALSE)
  }
  ajuste
}

modelos_tabla <- list(
  mutua = ajustar_tabla(
    Freq ~ zona_circulacion + tipo_vehiculo + siniestro
  ),
  conjunta = ajustar_tabla(
    Freq ~ zona_circulacion * tipo_vehiculo + siniestro
  ),
  condicional = ajustar_tabla(
    Freq ~ zona_circulacion * siniestro + tipo_vehiculo * siniestro
  ),
  homogenea = ajustar_tabla(
    Freq ~ (zona_circulacion + tipo_vehiculo + siniestro)^2
  )
)

# GL = celdas - parámetros. El saturado tiene un parámetro por celda,
# deviance = 0, y se calcula su AIC exactamente sin ajustar otro GLM.
resumen_tabla <- dplyr::bind_rows(lapply(names(modelos_tabla), function(nombre) {
  modelo <- modelos_tabla[[nombre]]
  gl <- stats::df.residual(modelo)
  tibble::tibble(
    modelo = nombre,
    gl = gl,
    deviance = stats::deviance(modelo),
    p_ajuste = if (gl > 0) {
      stats::pchisq(stats::deviance(modelo), df = gl, lower.tail = FALSE)
    } else NA_real_,
    AIC = stats::AIC(modelo)
  )
}))

n_celdas <- nrow(frecuencias_3v)
aic_saturado <- -2 * sum(stats::dpois(
  x = frecuencias_3v$Freq,
  lambda = frecuencias_3v$Freq,
  log = TRUE
)) + 2 * n_celdas

resumen_tabla <- dplyr::bind_rows(
  resumen_tabla,
  tibble::tibble(
    modelo = "saturado (referencia)",
    gl = 0L, deviance = 0, p_ajuste = NA_real_, AIC = aic_saturado
  )
)
resumen_tabla |>
  dplyr::mutate(dplyr::across(where(is.numeric), ~round(.x, 3)))

In [ ]:
# Contraste de la asociación zona-vehículo una vez condicionada
# por la presencia de siniestro, como en el bloque u22-lrt-jerarquia.
stats::anova(
  modelos_tabla$condicional,
  modelos_tabla$homogenea,
  test = "Chisq"
)

# La tabla describe proporciones; la tasa anual incorpora tiempo observado.
datos_07 |>
  dplyr::group_by(tipo_vehiculo, zona_circulacion) |>
  dplyr::summarise(
    polizas = dplyr::n(),
    porcentaje_con_siniestro = 100 * mean(n_siniestros > 0),
    tasa_anual = sum(n_siniestros) / sum(antiguedad),
    .groups = "drop"
  ) |>
  ggplot2::ggplot(ggplot2::aes(
    zona_circulacion, tipo_vehiculo, fill = tasa_anual
  )) +
  ggplot2::geom_tile(color = "white", linewidth = 1) +
  ggplot2::scale_fill_gradient(low = "grey95", high = "goldenrod") +
  ggplot2::labs(
    x = "Zona", y = "Vehículo", fill = "Partes/año",
    title = "Siniestralidad anual por vehículo y zona · Equipo 07"
  )

## 2.3 · Sobredispersión: Poisson, cuasi-Poisson y binomial negativa (P2)

Basado en la unidad 2.3: se reutiliza el Poisson de 2.1. Para Poisson Var(Y)=E(Y); cuasi-Poisson estima un factor φ; NB2 permite Var(Y)=μ+μ²/θ.

El AIC no es aplicable a cuasi-Poisson. Para elegir una tarifa hacen falta diagnóstico y validación fuera de muestra, no solo AIC.

In [ ]:
# Reutilizar la fórmula y el Poisson de 2.1.
m_siniestros_quasi <- stats::glm(
  formula_siniestros, family = stats::quasipoisson(), data = datos_07
)
m_siniestros_nb <- NULL
if (requireNamespace("MASS", quietly = TRUE)) {
  m_siniestros_nb <- tryCatch(
    MASS::glm.nb(formula_siniestros, data = datos_07),
    error = function(error) {
      message("No pudo ajustarse NB2: ", conditionMessage(error))
      NULL
    }
  )
} else message("MASS no disponible; se mantienen Poisson y cuasi-Poisson.")

tibble::tibble(
  modelo = c("Poisson", "Cuasi-Poisson", "Binomial negativa NB2"),
  dispersion_Pearson = c(
    dispersion_pearson(m_siniestros_pois),
    dispersion_pearson(m_siniestros_quasi),
    if (is.null(m_siniestros_nb)) NA_real_ else
      dispersion_pearson(m_siniestros_nb)
  ),
  AIC = c(
    stats::AIC(m_siniestros_pois), NA_real_,
    if (is.null(m_siniestros_nb)) NA_real_ else stats::AIC(m_siniestros_nb)
  )
)

In [ ]:
# Comparación observados frente a predichos según el ajuste.
comparacion_siniestros <- tibble::tibble(
  observado = datos_07$n_siniestros,
  poisson = stats::fitted(m_siniestros_pois),
  nb = if (is.null(m_siniestros_nb)) NA_real_ else
    stats::fitted(m_siniestros_nb)
)

comparacion_siniestros |>
  tidyr::pivot_longer(c(poisson, nb), names_to = "modelo",
                      values_to = "predicho") |>
  ggplot2::ggplot(ggplot2::aes(predicho, observado)) +
  ggplot2::geom_point(alpha = .18, color = "tomato") +
  ggplot2::geom_abline(slope = 1, intercept = 0,
                       linetype = "dashed", color = "goldenrod") +
  ggplot2::facet_wrap(~modelo, scales = "free_x") +
  ggplot2::labs(x = "Media predicha", y = "Recuento observado",
                title = "Calibración visual · Siniestros")

In [ ]:
# Diagnóstico adicional reproducible, solo si DHARMa está disponible.
if (requireNamespace("DHARMa", quietly = TRUE)) {
  set.seed(20252026)
  res_siniestros <- DHARMa::simulateResiduals(
    m_siniestros_pois, n = 250, plot = FALSE
  )
  plot(res_siniestros)
  print(DHARMa::testDispersion(res_siniestros))
  print(DHARMa::testZeroInflation(res_siniestros))
} else {
  message("DHARMa no disponible: el ajuste y los índices anteriores sí funcionan.")
}

## 2.4 · Demasiados ceros: fraude

**Adaptación de unidad_2_4.R:** u24-fraude-ceros, u24-zip, u24-zinb, u24-hurdle y u24-comparacion. Distinguimos **ceros esperados por el modelo de conteos** y **ceros adicionales**. ZIP añade una mezcla de ceros estructurales; hurdle modela una barrera para pasar de cero a positivo. No son la misma hipótesis.

⚠️ **Ejercicio complementario (RA4):** n_fraude estudia alertas de fraude, pero no es la respuesta permitida en las cinco preguntas del comité del informe ejecutivo. No emplear su resultado para recomendar tarifa de siniestros.

In [ ]:
formula_fraude <- n_fraude ~ potencia_cv + zona_circulacion +
  uso + tipo_vehiculo + offset(log(antiguedad))

m_fraude_pois <- stats::glm(
  formula_fraude, family = stats::poisson(), data = datos_07
)
m_fraude_nb <- if (requireNamespace("MASS", quietly = TRUE)) {
  MASS::glm.nb(formula_fraude, data = datos_07)
} else NULL

ceros_fraude <- tibble::tibble(
  origen = c("Observados", "Poisson", "Binomial negativa"),
  proporcion = c(
    mean(datos_07$n_fraude == 0),
    mean(stats::dpois(0, stats::fitted(m_fraude_pois))),
    if (is.null(m_fraude_nb)) NA_real_ else
      mean(stats::dnbinom(
        0, mu = stats::fitted(m_fraude_nb), size = m_fraude_nb$theta
      ))
  )
)
ceros_fraude

In [ ]:
# Los tres ajustes son opcionales si glmmTMB no está instalado.
m_fraude_zip <- NULL
m_fraude_zinb <- NULL
m_fraude_hurdle <- NULL

if (requireNamespace("glmmTMB", quietly = TRUE)) {
  m_fraude_zip <- glmmTMB::glmmTMB(
    formula_fraude, ziformula = ~uso + potencia_cv,
    family = stats::poisson(), data = datos_07
  )
  m_fraude_zinb <- glmmTMB::glmmTMB(
    formula_fraude, ziformula = ~uso + potencia_cv,
    family = glmmTMB::nbinom2(), data = datos_07
  )
  m_fraude_hurdle <- glmmTMB::glmmTMB(
    formula_fraude, ziformula = ~uso + potencia_cv,
    family = glmmTMB::truncated_poisson(), data = datos_07
  )
  print(summary(m_fraude_zip))
} else {
  message("Para ZIP/ZINB/hurdle falta glmmTMB: instalarlo en la primera sección.")
}

In [ ]:
modelos_fraude <- Filter(Negate(is.null), list(
  Poisson = m_fraude_pois, NB = m_fraude_nb,
  ZIP = m_fraude_zip, ZINB = m_fraude_zinb,
  Hurdle = m_fraude_hurdle
))
comparacion_fraude <- purrr::imap_dfr(modelos_fraude, function(modelo, nombre) {
  predicho <- as.numeric(stats::predict(modelo, type = "response"))
  tibble::tibble(
    modelo = nombre,
    AIC = stats::AIC(modelo),
    BIC = stats::BIC(modelo),
    RMSE = sqrt(mean((datos_07$n_fraude - predicho)^2))
  )
}) |>
  dplyr::arrange(AIC) |>
  dplyr::mutate(
    delta_AIC = AIC - min(AIC),
    peso_AIC = exp(-delta_AIC / 2) / sum(exp(-delta_AIC / 2))
  )
comparacion_fraude

# Los AIC comparan modelos ajustados a las mismas observaciones/respuesta.
# El RMSE dentro de muestra no prueba capacidad predictiva externa.

In [ ]:
# Cuánto se aparta cada modelo del número de ceros observado.
ceros_estimados <- purrr::imap_dfr(modelos_fraude, function(modelo, nombre) {
  p0 <- if (inherits(modelo, "glmmTMB")) {
    # En la distribución completa (también vale para el hurdle).
    zz <- stats::predict(modelo, type = "zprob")
    if (nombre == "Hurdle") zz
    else {
      mu <- stats::predict(modelo, type = "conditional")
      if (nombre == "ZINB") {
        theta <- stats::sigma(modelo)
        zz + (1 - zz) * stats::dnbinom(0, mu = mu, size = theta)
      } else zz + (1 - zz) * stats::dpois(0, mu)
    }
  } else if (inherits(modelo, "negbin")) {
    stats::dnbinom(0, mu = stats::fitted(modelo), size = modelo$theta)
  } else stats::dpois(0, stats::fitted(modelo))

  tibble::tibble(
    modelo = nombre,
    proporcion_ceros = mean(p0)
  )
})

ggplot2::ggplot(ceros_estimados,
                ggplot2::aes(modelo, proporcion_ceros)) +
  ggplot2::geom_col(fill = "goldenrod", color = "grey30") +
  ggplot2::geom_hline(
    yintercept = mean(datos_07$n_fraude == 0),
    linetype = "dashed", color = "tomato", linewidth = 1
  ) +
  ggplot2::coord_cartesian(ylim = c(0, 1)) +
  ggplot2::labs(x = "Modelo", y = "Proporción de ceros",
                title = "Ceros esperados vs observados (línea roja)")

## 2.5 · Región, agencia y efectos aleatorios

**Adaptación de unidad_2_5.R:** u25-glmm, u25-irr, u25-varcorr y u25-aic. La respuesta es **n_gestiones**. Las agencias están anidadas en regiones, así que examinamos la variación entre regiones y dentro de ellas, manteniendo el offset de exposición.

La estructura de agrupación se expresa con (1 | region/agencia). Un contraste de varianzas aleatorias en cero tiene condiciones de frontera: no interpretar automáticamente su p-valor como chi-cuadrado ordinario.

In [ ]:
tasas_agencia <- datos_07 |>
  dplyr::group_by(region, agencia) |>
  dplyr::summarise(
    polizas = dplyr::n(),
    gestiones = sum(n_gestiones),
    exposicion = sum(exposicion),
    tasa = gestiones / exposicion,
    .groups = "drop"
  )

ggplot2::ggplot(tasas_agencia,
                ggplot2::aes(x = region, y = tasa, color = region)) +
  ggplot2::geom_point(ggplot2::aes(size = polizas),
                      position = ggplot2::position_jitter(width = .15),
                      alpha = .75) +
  ggplot2::labs(x = "Región", y = "Gestiones por unidad de exposición",
                size = "Pólizas", title = "Heterogeneidad entre agencias") +
  ggplot2::theme(legend.position = "none")

In [ ]:
formula_gestiones <- n_gestiones ~ potencia_c + uso +
  offset(log(exposicion))

m_gestiones_pois <- stats::glm(
  formula_gestiones, family = stats::poisson(), data = datos_07
)
m_gestiones_nb <- if (requireNamespace("MASS", quietly = TRUE)) {
  MASS::glm.nb(formula_gestiones, data = datos_07)
} else NULL

m_gestiones_mixto <- NULL
if (requireNamespace("glmmTMB", quietly = TRUE)) {
  m_gestiones_mixto <- glmmTMB::glmmTMB(
    n_gestiones ~ potencia_c + uso + offset(log(exposicion)) +
      (1 | region/agencia),
    family = stats::poisson(), data = datos_07
  )
  print(summary(m_gestiones_mixto))
  print(c(
    convergencia = m_gestiones_mixto$fit$convergence == 0,
    hessiano_definido_positivo = m_gestiones_mixto$sdr$pdHess
  ))
} else {
  message("Modelo mixto pendiente: instalar glmmTMB.")
}

In [ ]:
modelos_gestiones <- Filter(Negate(is.null), list(
  Poisson = m_gestiones_pois,
  NB = m_gestiones_nb,
  GLMM = m_gestiones_mixto
))
comparacion_gestiones <- purrr::imap_dfr(
  modelos_gestiones, function(modelo, nombre) {
    tibble::tibble(
      modelo = nombre,
      AIC = stats::AIC(modelo),
      BIC = stats::BIC(modelo)
    )
  }
)
comparacion_gestiones

if (!is.null(m_gestiones_mixto)) {
  print(glmmTMB::VarCorr(m_gestiones_mixto))
  efectos_mixtos <- summary(m_gestiones_mixto)$coefficients$cond
  print(data.frame(
    termino = rownames(efectos_mixtos),
    IRR = exp(efectos_mixtos[, "Estimate"]),
    IC_inf = exp(efectos_mixtos[, "Estimate"] -
                   1.96 * efectos_mixtos[, "Std. Error"]),
    IC_sup = exp(efectos_mixtos[, "Estimate"] +
                   1.96 * efectos_mixtos[, "Std. Error"])
  ))
}

## 2.6 · Riesgo a trozos: adaptación a los datos del equipo 07

El guion **unidad_2_6.R** utiliza un panel póliza-año con n_partes_anual y expo_anual. **Nuestra cartera no contiene ese panel** y no podemos reconstruir una secuencia anual de siniestros a partir de un recuento agregado.

Sí disponemos de **t_primer_sin** (tiempo observado al primer siniestro o censura) y **evento** (indicador del suceso). Por tanto, podemos emplear un **modelo exponencial a trozos para el primer siniestro**: el mismo principio de dividir exposición temporal, pero otra variable objetivo. No estima reincidencias ni el historial de años posteriores.

El modelo requiere que los tiempos y la censura tengan el significado supuesto. Verificamos su coherencia antes de ajustarlo.

In [ ]:
if (!requireNamespace("survival", quietly = TRUE)) {
  message("Instala survival para ejecutar la unidad 2.6.")
} else {
  datos_evento <- datos_07 |>
    dplyr::filter(
      is.finite(t_primer_sin), t_primer_sin > 0,
      !is.na(evento), evento %in% c(0, 1),
      is.finite(antiguedad), t_primer_sin <= antiguedad
    )
  descarte_evento <- nrow(datos_07) - nrow(datos_evento)
  cat("Pólizas válidas:", nrow(datos_evento),
      "| excluidas por tiempo/censura:", descarte_evento, "\n")
  stopifnot(nrow(datos_evento) > 0)

  panel_primer_siniestro <- survival::survSplit(
    survival::Surv(t_primer_sin, evento) ~ .,
    data = datos_evento, cut = c(1, 3, 5),
    start = "t_inicio", end = "t_fin", episode = "trozo"
  ) |>
    dplyr::mutate(
      expo_tramo = t_fin - t_inicio,
      trozo = factor(trozo)
    ) |>
    dplyr::filter(expo_tramo > 0)

  stopifnot(
    all(panel_primer_siniestro$expo_tramo > 0),
    all(tapply(panel_primer_siniestro$evento,
               panel_primer_siniestro$id_poliza, sum) <= 1),
    sum(panel_primer_siniestro$evento) == sum(datos_evento$evento)
  )
  print(panel_primer_siniestro |>
    dplyr::group_by(trozo) |>
    dplyr::summarise(
      eventos = sum(evento),
      tiempo_observado = sum(expo_tramo),
      tasa = eventos / tiempo_observado,
      .groups = "drop"
    ))
}

In [ ]:
if (exists("panel_primer_siniestro") && requireNamespace("survival", quietly = TRUE)) {
  m_primer_siniestro <- stats::glm(
    evento ~ trozo + edad_conductor + potencia_cv +
      zona_circulacion + uso + offset(log(expo_tramo)),
    family = stats::poisson(), data = panel_primer_siniestro
  )
  print(resumen_coeficientes(m_primer_siniestro))

  tasa_tramos <- panel_primer_siniestro |>
    dplyr::group_by(trozo) |>
    dplyr::summarise(
      eventos = sum(evento),
      exposicion = sum(expo_tramo),
      tasa = eventos / exposicion,
      .groups = "drop"
    )

  print(
    ggplot2::ggplot(tasa_tramos, ggplot2::aes(trozo, tasa)) +
      ggplot2::geom_col(fill = "goldenrod", color = "grey35") +
      ggplot2::labs(x = "Tramo de antigüedad", y = "Tasa primer siniestro",
                    title = "Riesgo a trozos · Primer siniestro")
  )
} else {
  message("No se ha construido aún el panel de primer siniestro.")
}

## 2.7 · Resultados para el informe ejecutivo

**unidad_2_7.R** no aporta código de ajuste, sino un encargo de síntesis y evaluación. Esta última sección reúne las cifras **calculadas** y contrasta alternativas sin mezclar respuestas ni presentar relaciones causales sin diseño que las identifique.

**Lectura para el equipo:** ¿qué tasas son relevantes para la cartera?, ¿qué modelos muestran un ajuste más razonable?, ¿qué heterogeneidad aparece por segmento o agencia?, ¿qué incertidumbre o limitación condiciona cualquier recomendación?

In [ ]:
# Resumen consolidado, calculado a partir de los modelos existentes.
indicadores_07 <- tibble::tibble(
  indicador = c(
    "Pólizas", "Exposición total", "Tasa siniestros",
    "Tasa asistencias", "Tasa gestiones", "Proporción cero fraude",
    "Dispersión Poisson asistencias", "Dispersión Poisson siniestros"
  ),
  valor = c(
    nrow(datos_07),
    sum(datos_07$exposicion),
    sum(datos_07$n_siniestros) / sum(datos_07$exposicion),
    sum(datos_07$n_asistencia) / sum(datos_07$exposicion),
    sum(datos_07$n_gestiones) / sum(datos_07$exposicion),
    mean(datos_07$n_fraude == 0),
    dispersion_pearson(m_asistencia),
    dispersion_pearson(m_siniestros_pois)
  )
)
indicadores_07 |> dplyr::mutate(valor = round(valor, 4))

In [ ]:
# Contratos mínimos de la integración: ninguna conclusión usa el panel docente.
stopifnot(
  identical(nrow(datos_07), nrow(cartera)),
  all(is.finite(stats::fitted(m_asistencia))),
  all(stats::fitted(m_asistencia) > 0),
  all(is.finite(stats::fitted(m_siniestros_pois))),
  sum(tabla_3v) == nrow(datos_07),
  identical(nrow(resumen_respuestas), length(respuestas_07))
)
cat("Comprobaciones de estructura y coherencia: superadas.\n")
sessionInfo()